In [1]:
import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import time
import random
from tqdm import tqdm
import os
from matplotlib import font_manager as fm
import matplotlib.pyplot as plt

# ==================== 0. 字体与环境配置 ====================
def set_chinese_font():
    try:
        font_paths = ['/usr/share/fonts/truetype/wqy/wqy-microhei.ttc',
                      'C:/Windows/Fonts/simhei.ttf',
                      '/System/Library/Fonts/PingFang.ttc']
        for font_path in font_paths:
            if os.path.exists(font_path):
                fm.fontManager.addfont(font_path)
                plt.rcParams['font.family'] = fm.FontProperties(fname=font_path).get_name()
                plt.rcParams['axes.unicode_minus'] = False
                return True
    except: pass
    return False
set_chinese_font()

# ==================== 1. 卫星抗干扰仿真环境 ====================
class SatelliteEnvV3:
    def __init__(self, h5_path):
        print(f"📂 正在预载入数据集: {os.path.basename(h5_path)} ...")
        with h5py.File(h5_path, 'r') as f:
            self.pred_map = torch.FloatTensor(f['Y_horizon'][:]) 
            self.truth_map = torch.FloatTensor(f['Y_horizon'][:])
            self.type_data = torch.FloatTensor(f['gt_type'][:])
            
        self.num_channels = 10
        self.max_steps = len(self.pred_map)
        self.current_step = 0
        self.last_action = 0

    def reset(self):
        self.current_step = 0
        self.last_action = random.randint(0, 9)
        return self._get_state()

    def _get_state(self):
        map_feat = self.pred_map[self.current_step].flatten()
        type_feat = torch.tensor([self.type_data[self.current_step] / 4.0])
        return torch.cat([map_feat, type_feat])

    def step(self, action):
        is_collision = self.truth_map[self.current_step, 0, action] > 0.5
        future_risk = torch.mean(self.pred_map[self.current_step, :, action])
        
        if is_collision:
            reward = -100.0
        else:
            reward = 15.0 - (future_risk.item() * 30.0)
            if action == self.last_action:
                reward += 5.0 
            else:
                reward -= 2.0 
            
        self.last_action = action
        self.current_step += 1
        done = self.current_step >= self.max_steps - 1
        next_state = self._get_state() if not done else torch.zeros(101)
        return next_state, reward, done, is_collision

# ==================== 2. A2C Actor-Critic 网络架构 ====================
class ActorCriticNet(nn.Module):
    def __init__(self, input_dim=101, output_dim=10):
        super(ActorCriticNet, self).__init__()
        self.base = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU()
        )
        self.actor = nn.Sequential(nn.Linear(256, output_dim), nn.Softmax(dim=-1))
        self.critic = nn.Linear(256, 1)

    def forward(self, x):
        x = self.base(x)
        return self.actor(x), self.critic(x)

# ==================== 3. A2C 智能体 ====================
class A2CAgent:
    def __init__(self, state_dim=101, action_dim=10):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = ActorCriticNet(state_dim, action_dim).to(self.device)
        self.optimizer = optim.AdamW(self.model.parameters(), lr=1e-4)
        self.gamma = 0.98
        self.entropy_coef = 0.01 # 增加熵权重鼓励探索

    def choose_action(self, state):
        start_time = time.time()
        state = state.to(self.device).unsqueeze(0)
        probs, value = self.model(state)
        
        dist = Categorical(probs)
        action = dist.sample()
        
        latency = time.time() - start_time
        return action.item(), dist.log_prob(action), value, dist.entropy(), latency

    def update(self, log_prob, value, next_value, reward, done, entropy):
        # 计算 TD Error / Advantage: A(s) = r + gamma * V(s') - V(s)
        # 这里的 next_value 需要 detach，因为它是目标 
        target = reward + (1 - done) * self.gamma * next_value.detach()
        advantage = target - value
        
        # Actor Loss: -log_pi * Advantage
        actor_loss = -(log_prob * advantage.detach())
        # Critic Loss: MSE(V(s), Target)
        critic_loss = F.mse_loss(value, target)
        # Total Loss (增加熵项防止策略塌缩)
        loss = actor_loss + 0.5 * critic_loss - self.entropy_coef * entropy
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

# ==================== 4. 训练执行与指标报告 ====================
def run_a2c_training_with_metrics():
    DATA_PATH = "/root/autodl-tmp/validate/0218/Prediction/sim_dataset_v6_5/academic_long_horizon_v6_5_200k.h5"
    env = SatelliteEnvV3(DATA_PATH)
    agent = A2CAgent()
    
    episodes = 100
    metrics = {'reward': [], 'sr': [], 'hops': [], 'latency': [], 'throughput': [], 'stability': []}
    convergence_ep = -1

    print("🚀 启动 A2C (Advantage Actor-Critic) 离线训练...")
    for ep in range(episodes):
        state = env.reset()
        ep_reward, collisions, steps, hops = 0, 0, 0, 0
        ep_latencies, actions = [], []

        pbar = tqdm(total=env.max_steps, desc=f"Ep {ep+1}/{episodes}", leave=False)
        while True:
            # 1. 执行动作
            action, log_prob, value, entropy, lat = agent.choose_action(state)
            ep_latencies.append(lat)
            actions.append(action)

            if steps > 0 and action != env.last_action:
                hops += 1
                
            # 2. 与环境交互
            next_state, reward, done, collision = env.step(action)
            
            # 3. 获取下一状态价值并更新
            _, next_value = agent.model(next_state.to(agent.device).unsqueeze(0))
            agent.update(log_prob, value, next_value, reward, float(done), entropy)
                
            state = next_state
            ep_reward += reward
            if collision: collisions += 1
            steps += 1
            pbar.update(1)
            if done: break
        
        pbar.close()
        
        # 指标计算 
        sr = (1 - collisions / steps) * 100
        avg_hops = hops / (steps / 100)
        avg_lat = np.mean(ep_latencies) * 1000
        throughput = (1 - (collisions / steps)) * 1.0
        stability = np.std(actions)

        if convergence_ep == -1 and sr >= 95.0: convergence_ep = ep + 1

        metrics['reward'].append(ep_reward)
        metrics['sr'].append(sr)
        metrics['hops'].append(avg_hops)
        metrics['latency'].append(avg_lat)
        metrics['throughput'].append(throughput)
        metrics['stability'].append(stability)

        print(f"✅ Ep {ep+1} | 成功率: {sr:.2f}% | 跳频代价: {avg_hops:.1f} | 延迟: {avg_lat:.2f}ms | 吞吐量: {throughput:.2f}")

    print("\n" + "="*50)
    print("📊 A2C 离线实验对比报告总结")
    print("-" * 50)
    print(f"1. 收敛轮次 (Convergence Episode): {convergence_ep if convergence_ep != -1 else '未收敛'}")
    print(f"2. 平均推理时延 (Inference Latency): {np.mean(metrics['latency']):.4f} ms")
    print(f"3. 避障成功率峰值 (Peak Success Rate): {np.max(metrics['sr']):.2f} %")
    print(f"4. 稳态跳频代价 (Avg Switching Cost): {np.mean(metrics['hops'][-10:]):.2f} hops/100steps")
    print(f"5. 归一化吞吐量 (Throughput): {np.mean(metrics['throughput'][-10:]):.4f}")
    print(f"6. 动作稳定性 (Policy Stability Std): {np.mean(metrics['stability'][-10:]):.4f}")
    print("="*50)

    torch.save(agent.model.state_dict(), "best_a2c_offline_metrics.pth")
    return metrics

if __name__ == "__main__":
    run_a2c_training_with_metrics()

📂 正在预载入数据集: academic_long_horizon_v6_5_200k.h5 ...
🚀 启动 A2C (Advantage Actor-Critic) 离线训练...


✅ Ep 1 | 成功率: 82.22% | 跳频代价: 12.2 | 延迟: 0.94ms | 吞吐量: 0.82


✅ Ep 2 | 成功率: 82.85% | 跳频代价: 0.0 | 延迟: 0.93ms | 吞吐量: 0.83


✅ Ep 3 | 成功率: 82.85% | 跳频代价: 0.0 | 延迟: 0.92ms | 吞吐量: 0.83


✅ Ep 4 | 成功率: 82.85% | 跳频代价: 0.0 | 延迟: 0.90ms | 吞吐量: 0.83


...

✅ Ep 96 | 成功率: 82.85% | 跳频代价: 0.0 | 延迟: 0.60ms | 吞吐量: 0.83


✅ Ep 97 | 成功率: 82.85% | 跳频代价: 0.0 | 延迟: 0.61ms | 吞吐量: 0.83


✅ Ep 98 | 成功率: 82.85% | 跳频代价: 0.0 | 延迟: 0.63ms | 吞吐量: 0.83


✅ Ep 99 | 成功率: 82.85% | 跳频代价: 0.0 | 延迟: 0.62ms | 吞吐量: 0.83


✅ Ep 100 | 成功率: 82.85% | 跳频代价: 0.0 | 延迟: 0.62ms | 吞吐量: 0.83

📊 A2C 离线实验对比报告总结
--------------------------------------------------
1. 收敛轮次 (Convergence Episode): 未收敛
2. 平均推理时延 (Inference Latency): 0.7149 ms
3. 避障成功率峰值 (Peak Success Rate): 82.85 %
4. 稳态跳频代价 (Avg Switching Cost): 0.00 hops/100steps
5. 归一化吞吐量 (Throughput): 0.8285
6. 动作稳定性 (Policy Stability Std): 0.0000
